In [11]:
!curl -O https://www.antlr.org/download/antlr-4.13.2-complete.jar

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
 13  2.04M  13 288.0k   0      0 287.4k      0   00:07   00:01   00:06 288.0k
100  2.04M 100  2.04M   0      0  1.83M      0   00:01   00:01         288.0k
100  2.04M 100  2.04M   0      0  1.83M      0   00:01   00:01         288.0k
100  2.04M 100  2.04M   0      0  1.83M      0   00:01   00:01         288.0k


In [13]:
!pip install -q -U antlr4-python3-runtime


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
%%writefile FinanceLang.g4
grammar FinanceLang;

//PARSER
prog: stmt+ EOF;

stmt: CREAR OPERACION ID '=' expr ';' # regVariable
    | ELIMINAR OPERACION ID ';'       # delVariable
    | PROYECTAR ID A NUMBER MESES CON TASA expr ';' # projVariable
    | expr                            # printExpr
    ;

expr: '-' expr                        # exprNeg
    | expr ('*'|'/') expr             # exprMulDiv
    | expr ('+'|'-') expr             # exprAddSub
    | ID                              # exprId
    | NUMBER                          # exprNum
    | '(' expr ')'                    # exprParens
    ;

// --- LEXER ---

CREAR     : 'crear' ;
OPERACION : 'operacion' ;
ELIMINAR  : 'eliminar' ;
PROYECTAR : 'proyectar' ;
A         : 'a' ;
MESES     : 'meses' ;
CON       : 'con' ;
TASA      : 'tasa' ;

PLUS   : '+' ;
MINUS  : '-' ;
MUL    : '*' ;
DIV    : '/' ;
LPAREN : '(' ;
RPAREN : ')' ;
ASSIGN : '=' ;
SEMI   : ';' ;

ID     : [a-zA-Z_][a-zA-Z0-9_]* ;
NUMBER : [0-9]+ ('.' [0-9]+)? ;
WS     : [ \t\r\n]+ -> skip ;

Overwriting FinanceLang.g4


In [16]:
# 1. Generate the Python code (Using -jar makes this work on any OS)
!java -jar antlr-4.13.2-complete.jar FinanceLang.g4 -no-listener -visitor -Dlanguage=Python3

# 2. List the generated files (Using % instead of ! makes this work on any OS)
%ls FinanceLang*.py

 Volume in drive C is Windows
 Volume Serial Number is 80B8-DAD9

 Directory of c:\Users\marce\FinanceLang

10/05/2026  18:54             5,292 FinanceLangLexer.py
10/05/2026  18:54            22,548 FinanceLangParser.py
10/05/2026  18:54             2,373 FinanceLangVisitor.py
               3 File(s)         30,213 bytes
               0 Dir(s)  380,559,663,104 bytes free


In [17]:
test_input = """\
crear operacion capital = 1000 ;
crear operacion interes = 5.5 ;
crear operacion ganancia = capital + interes ;
crear operacion resultado = (capital * 2) + interes ;
eliminar operacion interes ;
proyectar capital a 12 meses con tasa 0.1 ;
proyectar resultado a 24 meses con tasa 0.5 ;
"""

with open("test_input.fin", "w") as f:
    f.write(test_input)

print("test_input.fin creado")

test_input.fin creado


In [18]:
from antlr4 import *
from FinanceLangLexer  import FinanceLangLexer
from FinanceLangParser import FinanceLangParser

# Preparar flujo de entrada
input_stream = FileStream("test_input.fin")
lexer        = FinanceLangLexer(input_stream)
stream       = CommonTokenStream(lexer)

# Crear el parser
parser = FinanceLangParser(stream)
tree   = parser.prog()

# Imprimir tokens
print("=== TOKENS ===")
stream.fill()
for tok in stream.tokens:
    if tok.type == Token.EOF:
        tipo = "EOF"
    elif tok.type < 0:
        tipo = "INVALID"
    else:
        tipo = FinanceLangParser.symbolicNames[tok.type]

    print(f"  {repr(tok.text):28} → {tipo}")

# Imprimir árbol sintáctico
print("\n=== ÁRBOL SINTÁCTICO ===")
print(tree.toStringTree(recog=parser))

=== TOKENS ===
  'crear'                      → CREAR
  'operacion'                  → OPERACION
  'capital'                    → ID
  '='                          → ASSIGN
  '1000'                       → NUMBER
  ';'                          → SEMI
  'crear'                      → CREAR
  'operacion'                  → OPERACION
  'interes'                    → ID
  '='                          → ASSIGN
  '5.5'                        → NUMBER
  ';'                          → SEMI
  'crear'                      → CREAR
  'operacion'                  → OPERACION
  'ganancia'                   → ID
  '='                          → ASSIGN
  'capital'                    → ID
  '+'                          → PLUS
  'interes'                    → ID
  ';'                          → SEMI
  'crear'                      → CREAR
  'operacion'                  → OPERACION
  'resultado'                  → ID
  '='                          → ASSIGN
  '('                          → LPAREN
  'capita

In [19]:
from FinanceLangVisitor import FinanceLangVisitor

class FinanceEvaluator(FinanceLangVisitor):
    def __init__(self):
        self.memoria = {}

    def visitPrintExpr(self, ctx: FinanceLangParser.PrintExprContext):
        val = self.visit(ctx.expr())
        if val is not None:
            print(f"Ans = {val:,.4f}")
        return None

    def visitExprNeg(self, ctx: FinanceLangParser.ExprNegContext):
        val = self.visit(ctx.expr())
        if val is None:
            return None
        return -val

    def visitRegVariable(self, ctx):
        nombre = ctx.ID().getText()
        valor  = self.visit(ctx.expr())
        if valor is None:
            print(f"[Error] No se pudo registrar '{nombre}': la expresion no es valida.")
            return None
        self.memoria[nombre] = valor
        print(f"[Registro] {nombre} establecido en {valor:,.2f}")
        return valor

    def visitDelVariable(self, ctx):
        nombre = ctx.ID().getText()
        if nombre in self.memoria:
            del self.memoria[nombre]
            print(f"[Eliminar] Operacion '{nombre}' borrada de la memoria.")
        else:
            print(f"[Error] No se puede eliminar '{nombre}': no existe.")
        return None

    def visitProjVariable(self, ctx):
        nombre = ctx.ID().getText()
        # ctx.NUMBER() devuelve el unico token NUMBER de la regla (los meses)
        # la tasa viene de ctx.expr() que puede ser un NUMBER o expresion
        meses  = float(ctx.NUMBER().getText())
        tasa   = self.visit(ctx.expr())

        if tasa is None:
            print(f"[Error] La tasa para '{nombre}' no es valida.")
            return None

        if tasa <= -1:
            print(f"[Error] Tasa {tasa} invalida: debe ser mayor a -1.")
            return None

        if nombre in self.memoria:
            valor_presente = self.memoria[nombre]
            # Formula de Interes Compuesto: VF = VP * (1 + i)^n
            valor_futuro = valor_presente * ((1 + tasa) ** meses)

            print(f"[Proyeccion] {nombre}:")
            print(f"   - Valor Inicial:    {valor_presente:,.2f}")
            print(f"   - Tiempo:           {int(meses)} meses")
            print(f"   - Tasa:             {tasa:.4f}")
            print(f"   - VALOR PROYECTADO: {valor_futuro:,.2f}")
            return valor_futuro

        print(f"[Error] Imposible proyectar: '{nombre}' no esta definido.")
        return None

    def visitExprMulDiv(self, ctx):
        izq      = self.visit(ctx.expr(0))
        der      = self.visit(ctx.expr(1))
        operador = ctx.getChild(1).getText()

        if izq is None or der is None:
            return None

        if operador == '*':
            return izq * der
        if operador == '/':
            if der == 0:
                print("[Error] Division por cero.")
                return None
            return izq / der
        return None

    def visitExprAddSub(self, ctx):
        izq      = self.visit(ctx.expr(0))
        der      = self.visit(ctx.expr(1))
        operador = ctx.getChild(1).getText()

        if izq is None or der is None:
            return None

        if operador == '+':
            return izq + der
        return izq - der

    def visitExprId(self, ctx):
        nombre = ctx.ID().getText()
        if nombre in self.memoria:
            return self.memoria[nombre]
        print(f"[Error] Variable '{nombre}' no definida.")
        return None

    def visitExprNum(self, ctx):
        return float(ctx.NUMBER().getText())

    def visitExprParens(self, ctx):
        return self.visit(ctx.expr())

In [20]:
# 1. Crear la instancia del evaluador
evaluador = FinanceEvaluator()

# 2. Visitar el árbol (tree es el resultado de parser.prog())
print("INICIO DE EJECUCIÓN FINANCIERA\n")
evaluador.visit(tree)
print("\nFIN DE LA SIMULACIÓN")

INICIO DE EJECUCIÓN FINANCIERA

[Registro] capital establecido en 1,000.00
[Registro] interes establecido en 5.50
[Registro] ganancia establecido en 1,005.50
[Registro] resultado establecido en 2,005.50
[Eliminar] Operacion 'interes' borrada de la memoria.
[Proyeccion] capital:
   - Valor Inicial:    1,000.00
   - Tiempo:           12 meses
   - Tasa:             0.1000
   - VALOR PROYECTADO: 3,138.43
[Proyeccion] resultado:
   - Valor Inicial:    2,005.50
   - Tiempo:           24 meses
   - Tasa:             0.5000
   - VALOR PROYECTADO: 33,760,812.01

FIN DE LA SIMULACIÓN


In [21]:
from antlr4 import *
from FinanceLangLexer  import FinanceLangLexer
from FinanceLangParser import FinanceLangParser

def ejecutar(codigo: str, titulo: str = ""):
    """Parsea y evalua un fragmento de FinanceLang desde un string."""
    print(f"\n{'='*55}")
    if titulo:
        print(f"  {titulo}")
    print(f"{'='*55}")

    stream = InputStream(codigo)
    lexer  = FinanceLangLexer(stream)
    lexer.removeErrorListeners()
    tokens = CommonTokenStream(lexer)
    parser = FinanceLangParser(tokens)
    parser.removeErrorListeners()
    tree   = parser.prog()

    ev = FinanceEvaluator()
    ev.visit(tree)
    print(f"{'─'*55}")
    return ev

##CASOS DE PRUEBA - POSITIVOS

In [22]:
# TC-01 | Flujo completo
ejecutar("""
crear operacion capital = 1000 ;
crear operacion interes = 5.5 ;
crear operacion ganancia = capital + interes ;
crear operacion resultado = (capital * 2) + interes ;
eliminar operacion interes ;
proyectar capital a 12 meses con tasa 0.1 ;
proyectar resultado a 24 meses con tasa 0.05 ;
""", "TC-01 | Flujo completo — crear, operar, eliminar, proyectar")


  TC-01 | Flujo completo — crear, operar, eliminar, proyectar
[Registro] capital establecido en 1,000.00
[Registro] interes establecido en 5.50
[Registro] ganancia establecido en 1,005.50
[Registro] resultado establecido en 2,005.50
[Eliminar] Operacion 'interes' borrada de la memoria.
[Proyeccion] capital:
   - Valor Inicial:    1,000.00
   - Tiempo:           12 meses
   - Tasa:             0.1000
   - VALOR PROYECTADO: 3,138.43
[Proyeccion] resultado:
   - Valor Inicial:    2,005.50
   - Tiempo:           24 meses
   - Tasa:             0.0500
   - VALOR PROYECTADO: 6,467.94
───────────────────────────────────────────────────────


In [23]:
# TC-02 | Descuento simple
ejecutar("""
crear operacion precio = 500 ;
crear operacion descuento = precio * 0.2 ;
crear operacion precio_final = precio - descuento ;
proyectar precio_final a 6 meses con tasa 0.08 ;
""", "TC-02 | Descuento y proyeccion simple")


  TC-02 | Descuento y proyeccion simple
[Registro] precio establecido en 500.00
[Registro] descuento establecido en 100.00
[Registro] precio_final establecido en 400.00
[Proyeccion] precio_final:
   - Valor Inicial:    400.00
   - Tiempo:           6 meses
   - Tasa:             0.0800
   - VALOR PROYECTADO: 634.75
───────────────────────────────────────────────────────


In [24]:
# TC-03 | Expresiones anidadas
ejecutar("""
crear operacion val_a = 100 ;
crear operacion val_b = 200 ;
crear operacion val_c = (val_a + val_b) * 2 ;
crear operacion val_d = val_c / 3 ;
proyectar val_d a 3 meses con tasa 0.01 ;
""", "TC-03 | Expresiones anidadas con parentesis")


  TC-03 | Expresiones anidadas con parentesis
[Registro] val_a establecido en 100.00
[Registro] val_b establecido en 200.00
[Registro] val_c establecido en 600.00
[Registro] val_d establecido en 200.00
[Proyeccion] val_d:
   - Valor Inicial:    200.00
   - Tiempo:           3 meses
   - Tasa:             0.0100
   - VALOR PROYECTADO: 206.06
───────────────────────────────────────────────────────


In [25]:
# TC-04 | Largo plazo
ejecutar("""
crear operacion fondo = 50000 ;
proyectar fondo a 60 meses con tasa 0.005 ;
""", "TC-04 | Proyeccion largo plazo — 60 meses, tasa 0.5% mensual")


  TC-04 | Proyeccion largo plazo — 60 meses, tasa 0.5% mensual
[Registro] fondo establecido en 50,000.00
[Proyeccion] fondo:
   - Valor Inicial:    50,000.00
   - Tiempo:           60 meses
   - Tasa:             0.0050
   - VALOR PROYECTADO: 67,442.51
───────────────────────────────────────────────────────


In [26]:
# TC-05 | Valor negativo
ejecutar("""
crear operacion deuda = 0 - 2000 ;
proyectar deuda a 12 meses con tasa 0.03 ;
""", "TC-05 | Valor negativo (deuda) con proyeccion")


  TC-05 | Valor negativo (deuda) con proyeccion
[Registro] deuda establecido en -2,000.00
[Proyeccion] deuda:
   - Valor Inicial:    -2,000.00
   - Tiempo:           12 meses
   - Tasa:             0.0300
   - VALOR PROYECTADO: -2,851.52
───────────────────────────────────────────────────────


## Casos de Prueba - Inválidos

In [27]:
# TC-E01 | Proyectar variable eliminada
ejecutar("""
crear operacion x = 100 ;
eliminar operacion x ;
proyectar x a 12 meses con tasa 0.1 ;
""", "TC-E01 | Proyectar variable eliminada")


  TC-E01 | Proyectar variable eliminada
[Registro] x establecido en 100.00
[Eliminar] Operacion 'x' borrada de la memoria.
[Error] Imposible proyectar: 'x' no esta definido.
───────────────────────────────────────────────────────


In [28]:
# TC-E02 | Variable no definida en expresion
ejecutar("""
crear operacion resultado = capital + 500 ;
""", "TC-E02 | Variable no definida en expresion")


  TC-E02 | Variable no definida en expresion
[Error] Variable 'capital' no definida.
[Error] No se pudo registrar 'resultado': la expresion no es valida.
───────────────────────────────────────────────────────


In [29]:
# TC-E03 | Division por cero
ejecutar("""
crear operacion base = 100 ;
crear operacion division_cero = base / 0 ;
""", "TC-E03 | Division por cero")


  TC-E03 | Division por cero
[Registro] base establecido en 100.00
[Error] Division por cero.
[Error] No se pudo registrar 'division_cero': la expresion no es valida.
───────────────────────────────────────────────────────


In [30]:
# TC-E04 | Eliminar variable inexistente
ejecutar("""
eliminar operacion no_existe ;
""", "TC-E04 | Eliminar variable inexistente")


  TC-E04 | Eliminar variable inexistente
[Error] No se puede eliminar 'no_existe': no existe.
───────────────────────────────────────────────────────


In [31]:
# TC-E05 | Tasa invalida
ejecutar("""
crear operacion valor = 500 ;
proyectar valor a 12 meses con tasa 0 - 1.5 ;
""", "TC-E05 | Tasa invalida menor o igual a -1")


  TC-E05 | Tasa invalida menor o igual a -1
[Registro] valor establecido en 500.00
[Error] Tasa -1.5 invalida: debe ser mayor a -1.
───────────────────────────────────────────────────────


In [32]:
# TC-E06 | Cadena de error
ejecutar("""
crear operacion resultado_mal = indefinida * 2 ;
proyectar resultado_mal a 6 meses con tasa 0.1 ;
""", "TC-E06 | Cadena de error — variable indefinida bloquea registro y proyeccion")


  TC-E06 | Cadena de error — variable indefinida bloquea registro y proyeccion
[Error] Variable 'indefinida' no definida.
[Error] No se pudo registrar 'resultado_mal': la expresion no es valida.
[Error] Imposible proyectar: 'resultado_mal' no esta definido.
───────────────────────────────────────────────────────


In [34]:
# Estado de la memoria al terminar la ejecucion principal
print("\nEstado final de memoria (ejecucion principal):")
print("─" * 40)
for k, v in evaluador.memoria.items():
    print(f"  {k:<20} = {v:>12,.2f}")
if not evaluador.memoria:
    print("  (vacia)")


Estado final de memoria (ejecucion principal):
────────────────────────────────────────
  capital              =     1,000.00
  ganancia             =     1,005.50
  resultado            =     2,005.50
